# NOA

This notebook provides wrapper functions for calling the NOA (Naive Online ALignment) algorithm. Running this algorithm requires installing some other software, which is described below. This notebook implements the `offline_processing()` and `online_processing()` functions, which will be imported and run in `02_RunExperiment.ipynb`.

## Offline Processing

In the offline processing stage, three things are computed and stored in the `cache/` folder:
- Chroma STFT features for the orchestra recording
- Chroma STFT features for the full mix recording

In [ ]:
import os
import numpy as np
import librosa as lb

import system_utils
from noa import alignNOA, compute_cosine_distance, compute_euclidean_distance

In [ ]:
def offline_processing(scenario_dir, cache_dir, hop_length):
    '''
    Carries out offline processing for a simple offline DTW system.
    
    Inputs
    scenario_dir: The scenario directory to process
    cache_dir: The location of the cache directory
    hop_length: The hop length in samples used when computing chroma features
    
    This function will store the computed chroma features and estimated alignment in the cache folder.
    '''
    
    # setup
    system_utils.verify_scenario_dir(scenario_dir)
    if os.path.exists(cache_dir):
        # print(f'{cache_dir} has already been processed.  Skipping.')
        pass
    else:
        # setup
        os.makedirs(cache_dir)

    save_dir = f'{cache_dir}/pref_stft.npy'
    
    if os.path.exists(save_dir):
        print(f'{save_dir} already exists.  Skipping.')
        return

    # compute piano solo reference features
    pref_file = f'{scenario_dir}/pref.wav'
    
    y_pref, sr = lb.load(pref_file)
    F_pref = lb.feature.chroma_stft(y=y_pref, sr=sr, hop_length=hop_length, center=False)

    # save to cache
    np.save(save_dir, F_pref)
    
    return

In [ ]:
def verify_cache_dir(indir):
    """
    """
    assert os.path.exists(f'{indir}/pref_stft.npy'), f'pref_stft.npy missing from {indir}'

## Online Processing

In the online processing stage, the following steps are done:
- compute alignment between P_query and P_ref and thus O

### Wrapper Implementation

In [ ]:
def online_processing(scenario_dir, out_dir, cache_dir, hop_length, monotonic=False):
    # verify & setup
    system_utils.verify_scenario_dir(scenario_dir)
    verify_cache_dir(cache_dir)
    assert not os.path.exists(out_dir), f'Output directory {out_dir} already exists.'
    os.makedirs(out_dir)
    
    # compute features
    p_file = f'{scenario_dir}/p.wav'
    

    y, sr = lb.core.load(p_file)
    F_p = lb.feature.chroma_stft(y=y, sr=sr, hop_length=hop_length, center=False)
    F_pref = np.load(f'{cache_dir}/pref_stft.npy')

    hop_sec = hop_length / sr

    # get ref start idx
    info_file = f'{scenario_dir}/scenario.info'
    with open(info_file, 'r') as f:
        info = f.read().split()
    ref_start_time = float(info[-4])
    
    # compute alignment - cosine for librosa features
    cost_metric = compute_cosine_distance
    wp = alignNOA(F_p, F_pref, ref_start_time=ref_start_time, cost_metric=cost_metric, monotonic=monotonic)
    
    np.save(f'{out_dir}/hyp.npy', wp)
    return